# Clasificacion del resultado de servicios usando CRISP-DM

Esta libreta documenta una solucion de **clasificacion multiclase**
para pronosticar el resultado final de una orden.


## 1. Comprension del negocio

- Solucion: clasificacion multiclase con `LogisticRegression`.
- Objetivo: pronosticar si una orden termina como
  `completed_on_time`, `delayed` o `cancelled`.
- Unidad de analisis: orden historica de servicio.


## 2. Comprension de los datos

Primero se carga el dataset producido por ETL y se revisa su
estructura general.


In [1]:
import os
import json
import matplotlib.pyplot as plt
import pickle
import unicodedata
from pathlib import Path

os.environ["LOKY_MAX_CPU_COUNT"] = "1"

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def normalize_text(value, fallback):
    if pd.isna(value):
        return fallback
    text = str(value).strip().lower()
    text = "".join(
        character
        for character in unicodedata.normalize("NFD", text)
        if unicodedata.category(character) != "Mn"
    )
    text = " ".join(text.split())
    return text or fallback

BACKEND = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "package.json").exists() and (path / "src").exists()
)
RANDOM_STATE = 42


Librerias cargadas correctamente.
Semilla fija: 42


In [2]:
df = pd.read_csv(BACKEND / "05_Datasets" / "03_clasificacion_resultado_servicios.csv")
print(df.head().to_string(index=False))


order_id  source_folio  is_synthetic         source_created_at         source_updated_at  promised_lead_hours  registration_hour  registration_weekday  item_count  total_quantity  distinct_study_count  package_component_count  subtotal_amount  courtesy_percent  discount_amount  total_amount     branch_name dominant_price_type           outcome
     663 ECO-ML-000137          True 2025-11-01T14:07:00+00:00 2025-11-01T16:07:00+00:00                 20.0           8.116667                     6           6               6                     6                        0           3683.5               5.0           184.18       3499.32    sucursal sur             special         cancelled
     903 ECO-ML-000377          True 2025-11-01T14:07:00+00:00 2025-11-04T01:13:00+00:00                 60.0           8.116667                     6           6               6                     6                        0           3936.0               5.0           196.80       3739.20    sucursal sur 

In [3]:
print(df.shape)


(2000, 19)


In [4]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   order_id                 2000 non-null   int64  
 1   source_folio             2000 non-null   str    
 2   is_synthetic             2000 non-null   bool   
 3   source_created_at        2000 non-null   str    
 4   source_updated_at        2000 non-null   str    
 5   promised_lead_hours      2000 non-null   float64
 6   registration_hour        2000 non-null   float64
 7   registration_weekday     2000 non-null   int64  
 8   item_count               2000 non-null   int64  
 9   total_quantity           2000 non-null   int64  
 10  distinct_study_count     2000 non-null   int64  
 11  package_component_count  2000 non-null   int64  
 12  subtotal_amount          2000 non-null   float64
 13  courtesy_percent         2000 non-null   float64
 14  discount_amount          2000 non-n

In [5]:
print(df[["promised_lead_hours", "item_count", "total_amount"]].describe().round(2).to_string())


       promised_lead_hours  item_count  total_amount
count              2000.00     2000.00       2000.00
mean                 41.64        3.44       1868.58
std                  18.84        1.77        977.91
min                   1.00        1.00         63.75
25%                  26.00        2.00        995.90
50%                  42.00        3.00       1830.80
75%                  58.00        5.00       2731.52
max                  84.00        6.00       4291.50


In [6]:
print(
    df[
        [
            "promised_lead_hours",
            "registration_hour",
            "item_count",
            "branch_name",
            "dominant_price_type",
            "outcome",
        ]
    ].isnull().sum().to_string()
)


promised_lead_hours    0
registration_hour      0
item_count             0
branch_name            0
dominant_price_type    0
outcome                0


### Control de calidad inicial

Las salidas anteriores documentan:

- muestra inicial
- dimensiones
- tipos de dato
- estadisticos descriptivos
- valores faltantes

Durante la preparacion tambien se depuran registros invalidos y se
ordena el historial en forma temporal.


### Descripcion de las variables

Algunas variables utilizadas son:

- `promised_lead_hours`
- `registration_hour`
- `item_count`
- `total_quantity`
- `subtotal_amount`
- `total_amount`
- `branch_name`
- `dominant_price_type`


## 3. Preparacion de los datos

Se convierten tipos de datos, se ordena el historial por fecha y se
define un holdout temporal: pasado para train y tramo mas reciente
para test.


In [7]:
work = df.copy()
work["order_id"] = pd.to_numeric(work["order_id"], errors="coerce")
work["source_created_at"] = pd.to_datetime(work["source_created_at"], errors="coerce")

numeric_features = [
    "promised_lead_hours",
    "registration_hour",
    "registration_weekday",
    "item_count",
    "total_quantity",
    "distinct_study_count",
    "package_component_count",
    "subtotal_amount",
    "courtesy_percent",
    "discount_amount",
    "total_amount",
]
categorical_features = ["branch_name", "dominant_price_type"]

for column in numeric_features:
    work[column] = pd.to_numeric(work[column], errors="coerce")
for column in categorical_features:
    work[column] = work[column].map(lambda value: normalize_text(value, "desconocido"))

work = work.dropna(subset=["order_id", "source_created_at", "outcome"])
work = work[work["outcome"].isin(["completed_on_time", "delayed", "cancelled"])]
work = work[work["promised_lead_hours"] > 0]
work = work[work["item_count"] > 0]
work = work.drop_duplicates(subset=["order_id"], keep="first")
work = work.sort_values(["source_created_at", "order_id"]).reset_index(drop=True)

features = [*numeric_features, *categorical_features]
test_size = max(1, int(round(len(work) * 0.20)))
train_df = work.iloc[:-test_size].copy()
test_df = work.iloc[-test_size:].copy()

print("Filas utiles:", len(work))
print("Filas train:", len(train_df))
print("Filas test:", len(test_df))


Filas utiles: 2000
Filas train: 1600
Filas test: 400


### Grafica de distribucion de clases

Esta grafica ayuda a revisar si las tres clases tienen una
presencia razonable dentro del dataset util.


In [8]:
distribution = work["outcome"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(distribution.index, distribution.values, color=["#4e79a7", "#f28e2b", "#e15759"])
ax.set_title("Clasificacion: distribucion de clases")
ax.set_xlabel("Clase")
ax.set_ylabel("Numero de ordenes")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 4. Modelado y entrenamiento

Se construye un `Pipeline` con preprocesamiento y una
`LogisticRegression` multiclase.

Para reproducibilidad se usa `RANDOM_STATE = 42`.


In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", make_onehot_encoder()),
                ]
            ),
            categorical_features,
        ),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                solver="lbfgs",
            ),
        ),
    ]
)

model.fit(train_df[features], train_df["outcome"])
print("Clasificador entrenado correctamente.")


Clasificador entrenado correctamente.


## 5. Evaluacion

Se calcula la exactitud y el reporte por clase, ademas de guardar
las probabilidades del conjunto de prueba.


In [10]:
predicted = model.predict(test_df[features])
probabilities = model.predict_proba(test_df[features])
classes = list(model.named_steps["model"].classes_)

accuracy = accuracy_score(test_df["outcome"], predicted)
report = classification_report(
    test_df["outcome"],
    predicted,
    output_dict=True,
    zero_division=0,
)

print(
    json.dumps(
        {
            "accuracy": round(float(accuracy), 4),
            "report": report,
        },
        indent=2,
        ensure_ascii=False,
    )
)


{
  "accuracy": 0.7965,
  "report": {
    "precision": {
      "cancelled": 0.6415,
      "completed_on_time": 0.8816,
      "delayed": 0.7009,
      "accuracy": 0.7965,
      "macro avg": 0.7413,
      "weighted avg": 0.802
    },
    "recall": {
      "cancelled": 0.9189,
      "completed_on_time": 0.8553,
      "delayed": 0.6508,
      "accuracy": 0.7965,
      "macro avg": 0.8083,
      "weighted avg": 0.7965
    },
    "f1-score": {
      "cancelled": 0.7556,
      "completed_on_time": 0.8683,
      "delayed": 0.6749,
      "accuracy": 0.7965,
      "macro avg": 0.7662,
      "weighted avg": 0.7966
    },
    "support": {
      "cancelled": 37.0,
      "completed_on_time": 235.0,
      "delayed": 126.0,
      "accuracy": 0.7965,
      "macro avg": 398.0,
      "weighted avg": 398.0
    }
  }
}


### Interpretacion de la evaluacion

La exactitud resume el desempeno general, pero el reporte por clase
permite revisar si el modelo distingue ordenes a tiempo,
retrasadas y canceladas. Las probabilidades ayudan a justificar
cada prediccion individual.


In [11]:
matrix = confusion_matrix(test_df["outcome"], predicted, labels=classes)

fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(matrix, cmap="Blues")
ax.set_title("Clasificacion: matriz de confusion")
ax.set_xlabel("Prediccion")
ax.set_ylabel("Valor real")
ax.set_xticks(range(len(classes)))
ax.set_xticklabels(classes, rotation=20)
ax.set_yticks(range(len(classes)))
ax.set_yticklabels(classes)

for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, int(matrix[i, j]), ha="center", va="center", color="#111111")

plt.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


In [12]:
predicciones = test_df[["order_id", "source_created_at", "outcome"]].copy()
predicciones["predicted_outcome"] = predicted
predicciones["confidence"] = np.round(probabilities.max(axis=1), 4)

for class_name in classes:
    predicciones[f"probability_{class_name}"] = np.round(
        probabilities[:, classes.index(class_name)],
        4,
    )

print(predicciones.head(8).to_string(index=False))


order_id         source_created_at           outcome predicted_outcome  confidence  probability_cancelled  probability_completed_on_time  probability_delayed
    2179 2026-05-31 12:59:09+00:00           delayed           delayed      0.9609                 0.0111                         0.0280               0.9609
    2180 2026-05-31 17:33:19+00:00         cancelled         cancelled      0.8350                 0.8350                         0.0062               0.1588
    2181 2026-05-31 21:30:29+00:00 completed_on_time completed_on_time      0.9951                 0.0000                         0.9951               0.0049
     530 2026-05-31 21:44:00+00:00           delayed           delayed      0.5976                 0.0601                         0.3422               0.5976
     770 2026-05-31 21:44:00+00:00           delayed           delayed      0.5459                 0.4239                         0.0302               0.5459
    1010 2026-05-31 21:44:00+00:00           delayed

## 6. Exportacion del modelo

El modelo entrenado se guarda en
`07_Modelos/classification_service_outcome_model.pkl`.
Ese mismo nombre es el que espera el flujo Python del sistema web
en `ml-artifacts/scripts/classification_model.py`.


In [13]:
model_relative = Path("07_Modelos") / "classification_service_outcome_model.pkl"
predictions_relative = Path("05_Datasets") / "03_clasificacion_resultado_servicios_predicciones_test.csv"
model_path = BACKEND / model_relative
predictions_path = BACKEND / predictions_relative

bundle = {
    "model": model,
    "features": features,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "classes": classes,
}

with model_path.open("wb") as file:
    pickle.dump(bundle, file)

predicciones.to_csv(predictions_path, index=False)

print("Modelo exportado en:", model_relative.as_posix())
print("Predicciones test guardadas en:", predictions_relative.as_posix())


Modelo exportado en: 07_Modelos/classification_service_outcome_model.pkl
Predicciones test guardadas en: 05_Datasets/03_clasificacion_resultado_servicios_predicciones_test.csv


## 7. Ejemplo de prediccion

Se realiza una prediccion individual usando el modelo ya entrenado.


In [14]:
muestra = pd.DataFrame(
    [
        {
            "promised_lead_hours": work.iloc[0]["promised_lead_hours"],
            "registration_hour": work.iloc[0]["registration_hour"],
            "registration_weekday": work.iloc[0]["registration_weekday"],
            "item_count": work.iloc[0]["item_count"],
            "total_quantity": work.iloc[0]["total_quantity"],
            "distinct_study_count": work.iloc[0]["distinct_study_count"],
            "package_component_count": work.iloc[0]["package_component_count"],
            "subtotal_amount": work.iloc[0]["subtotal_amount"],
            "courtesy_percent": work.iloc[0]["courtesy_percent"],
            "discount_amount": work.iloc[0]["discount_amount"],
            "total_amount": work.iloc[0]["total_amount"],
            "branch_name": work.iloc[0]["branch_name"],
            "dominant_price_type": work.iloc[0]["dominant_price_type"],
        }
    ]
)

predicted_class = model.predict(muestra)[0]
predicted_proba = model.predict_proba(muestra)[0]

print("Entrada:")
print(json.dumps(muestra.iloc[0].to_dict(), indent=2, ensure_ascii=False))
print("\nPrediccion:")
print(
    json.dumps(
        {
            "predicted_outcome": predicted_class,
            "confidence": round(float(predicted_proba.max()), 4),
            "probabilities": {
                class_name: round(float(predicted_proba[classes.index(class_name)]), 4)
                for class_name in classes
            },
        },
        indent=2,
        ensure_ascii=False,
    )
)


Entrada:
{
  "promised_lead_hours": 20.0,
  "registration_hour": 8.116666666666665,
  "registration_weekday": 6.0,
  "item_count": 6.0,
  "total_quantity": 6.0,
  "distinct_study_count": 6.0,
  "package_component_count": 0.0,
  "subtotal_amount": 3683.5,
  "courtesy_percent": 5.0,
  "discount_amount": 184.18,
  "total_amount": 3499.32,
  "branch_name": "sucursal sur",
  "dominant_price_type": "special"
}

Predicción:
{
  "predicted_outcome": "cancelled",
  "confidence": 0.7893,
  "probabilities": {
    "cancelled": 0.7893,
    "completed_on_time": 0.0015,
    "delayed": 0.2091
  }
}
